# Файлы и форматы данных

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи самостоятельно, затем обратитесь к разделу решений.

## Уроки по этим темам

Материал этого ноутбука можно изучать по отдельным урокам:

- [Текстовые файлы и пути](../../python/functions-files/text-files/)
- [CSV, JSON и проверка данных](../../python/functions-files/csv-json/)


## Чему научимся

Читать текст, CSV и JSON, проверять поля и сообщать номер повреждённой строки. Используем стандартную библиотеку: дополнительных пакетов не нужно.

## Пути и текст

Модуль — набор готовых функций и типов; `import` делает его доступным программе. `Path` из модуля `pathlib` представляет путь. Относительный путь отсчитывается от текущей рабочей папки, которая не всегда совпадает с папкой ноутбука. `Path.cwd()` показывает её. Оператор `/` для путей добавляет часть пути.

Текстовые файлы хранят байты; кодировка объясняет, как превратить их в символы. Для наших файлов явно задаём UTF-8. Режим `"r"` читает, `"w"` записывает и стирает прежнее содержимое. Конструкция `with ... as stream` открывает ресурс и даёт ему имя `stream` внутри блока; по окончании блока файл закрывается даже при исключении. `path.open(...)` открывает файл, `stream.write(text)` записывает текст, `stream.read()` читает весь текст. Последовательность `\n` внутри строки обозначает перевод строки.

`from module import name` подключает только указанное имя из модуля. `TemporaryDirectory` из `tempfile` создаёт временную папку; `with TemporaryDirectory() as folder` даёт её путь и удаляет папку при выходе из блока.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    path = Path(folder) / "note.txt"
    with path.open("w", encoding="utf-8") as stream:
        stream.write("Размах: 22 см\n")
    with path.open(encoding="utf-8") as stream:
        text = stream.read()
    assert text.strip() == "Размах: 22 см"
print(text.strip())  # Размах: 22 см


Пример не перезаписывает ученические файлы. Для большого файла перебирайте строки `for line in stream`, а не загружайте весь текст через `read`.

## CSV: строки таблицы

CSV хранит таблицу с разделителями и правилами кавычек. Например, файл с двумя полями может выглядеть так:

```text
id,span_cm
a,12
b,22
```

Первая строка — заголовок. Не разбивайте строки вручную по запятой: запятая может быть частью поля в кавычках. `csv.DictReader` читает заголовок и выдаёт строки как словари. Значения пока остаются строками: `"22"` нужно превратить в число.

**Схема данных** — договорённость об именах полей, их типах и допустимых значениях. В нашем файле нужны поля `id` и `span_cm` в этом порядке, непустой уникальный идентификатор и положительный конечный размах.

## Разобранный пример: проверяем измерения

Сначала опишем чтение отдельной функцией. У `DictReader` свойство `fieldnames` хранит названия столбцов. Если полей в строке слишком много, лишние попадут под ключ `None`; если слишком мало — у недостающего поля значение будет `None`. `row.values()` позволяет проверить значения словаря. Проверку числа полей выполняем **до** обращения к конкретному полю.

`enumerate(..., start=2)` нумерует записи после заголовка, начиная с 2. В нашем простом CSV каждая запись занимает одну строку. `float` принимает и специальные строки `"nan"` и `"inf"`: неопределённое числовое значение и бесконечность. `math.isfinite` возвращает `True` только для конечного числа, исключая оба случая. Пустое поле не равно нулю: отсутствие измерения нельзя автоматически заменять нулевой массой или размахом.

In [ ]:
import csv
import math

def read_measurements(path):
    rows = []
    seen = set()
    with path.open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["id", "span_cm"]:
            raise ValueError("Ожидались поля id, span_cm в этом порядке")
        for line_number, row in enumerate(reader, start=2):
            if None in row or None in row.values():
                raise ValueError(f"Запись {line_number}: неверное число полей")
            identifier = row["id"].strip()
            try:
                span = float(row["span_cm"])
            except ValueError:
                raise ValueError(f"Запись {line_number}: размах не число")
            if not identifier or identifier in seen:
                raise ValueError(f"Запись {line_number}: пустой или повторный id")
            if not math.isfinite(span) or span <= 0:
                raise ValueError(f"Запись {line_number}: нужен конечный размах > 0")
            seen.add(identifier)
            rows.append({"id": identifier, "span_cm": span})
    return rows


Проследите одну запись: `"12"` превращается в `12.0`, проходит проверку знака, затем попадает в результат. Строка с `"nan"` преобразуется без ошибки, но будет отклонена проверкой `isfinite`. При нарушении схемы функция останавливается с объяснением; строки не исчезают молча.

Теперь создадим файл и применим функцию. Это два вымышленных измерения для обучения работе с форматом, а не реальные полёты. `csv.DictWriter` записывает словари по заданным `fieldnames`; `writeheader` пишет заголовок, `writerows` — строки. `newline=""` оставляет обработку переводов строк модулю CSV и используется и при записи, и при чтении.

In [ ]:
with TemporaryDirectory() as folder:
    path = Path(folder) / "measurements.csv"
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["id", "span_cm"])
        writer.writeheader()
        writer.writerows([
            {"id": "a", "span_cm": 12},
            {"id": "b", "span_cm": 22},
        ])
    rows = read_measurements(path)
assert rows == [{"id": "a", "span_cm": 12.0}, {"id": "b", "span_cm": 22.0}]
print(rows)


## JSON: вложенные данные

JSON хранит объекты, массивы, строки, числа, логические значения и `null`. В Python объект JSON читается как словарь, массив как список, `null` как `None`. `json.dump` записывает в файл, `json.load` читает; варианты `dumps` и `loads` работают со строкой.

`ensure_ascii=False` сохраняет кириллицу читаемой; `allow_nan=False` запрещает запись нечисловых значений и бесконечностей как нестандартного JSON.

In [ ]:
import json

config = {"seed": 17, "features": ["span_cm"], "comment": "учебный опыт"}
encoded = json.dumps(config, ensure_ascii=False, allow_nan=False)
restored = json.loads(encoded)
assert restored == config


Успешное чтение JSON ещё не проверяет смысл: отдельно проверяйте обязательные ключи, типы и диапазоны. Повреждённый JSON вызывает `json.JSONDecodeError`.

## Задачи

### 1. Базовая

Откуда отсчитывается путь `Path("data.csv")`?

<details>
<summary>Подсказка</summary>

Вспомните `Path.cwd()`.

</details>



### 2. Базовая

Чем отличаются JSON `null`, число 0 и строка "0" после чтения Python?

<details>
<summary>Подсказка</summary>

У значений разные типы.

</details>



### 3. Прикладная

Проверьте значения размаха `"12.5"`, `""`, `"nan"`, `"-2"`. Допустимо только первое.

<details>
<summary>Подсказка</summary>

Сначала преобразуйте, затем проверьте конечность и знак.

</details>



### 4. Прикладная

Сохраните словарь `{"name": "крыло", "count": 2}` в JSON во временной папке и прочитайте обратно.

<details>
<summary>Подсказка</summary>

Один файл открывается сначала для записи, затем для чтения.

</details>



### 5. Олимпиадная

Почему `line.split(",")` неверно разберёт CSV-строку `a,"крыло, большое",22`?

<details>
<summary>Подсказка</summary>

Разделитель внутри кавычек относится к значению.

</details>



### 6. Олимпиадная

Две строки имеют один `id` и разные массы. Почему проверка типов не обнаруживает проблему?

<details>
<summary>Подсказка</summary>

Схема и согласованность записей — разные проверки.

</details>



## Итоги и самопроверка

Сначала объясните схему файла, затем напишите проверки. Умейте отличить ошибку пути, кодировки, синтаксиса и смысла данных. В ноутбуке все файлы создаются и удаляются во временной папке.


## Решения задач

### Задача 1

От текущей рабочей папки процесса, а не обязательно от папки ноутбука.

### Задача 2

Они превращаются в `None`, целое число `0` и строку `"0"`.

### Задача 3

In [ ]:
def valid_span(text):
    try:
        value = float(text)
    except ValueError:
        return False
    return math.isfinite(value) and value > 0

assert [valid_span(s) for s in ["12.5", "", "nan", "-2"]] == [True, False, False, False]


### Задача 4

In [ ]:
with TemporaryDirectory() as folder:
    path = Path(folder) / "config.json"
    original = {"name": "крыло", "count": 2}
    with path.open("w", encoding="utf-8") as stream:
        json.dump(original, stream, ensure_ascii=False)
    with path.open(encoding="utf-8") as stream:
        assert json.load(stream) == original


### Задача 5

Ручное разбиение даст четыре части вместо трёх. CSV-парсер учтёт кавычки и сохранит «крыло, большое» одним полем.

### Задача 6

Оба числа могут быть корректны. Нужно проверить уникальность идентификатора и выяснить смысл повтора: ошибка это или повторное измерение. Молча оставить одну строку нельзя.